# CTDenoiser — Build HDF5 Cache

**Run this notebook once** to download the TCIA LDCT dataset and build `ldct_cache.h5` on Google Drive.
After this notebook completes, open `ctdenoiser_colab.ipynb` for training.

```
ldct_cache.h5
  L004_low    (num_slices, 512, 512)  float32  raw HU
  L004_full   (num_slices, 512, 512)  float32  raw HU
  ...
```

**Expected runtime:** 30–90 min &nbsp;|&nbsp; **Disk:** ~4 GB per patient pair

> Re-running this notebook when the cache already exists on Drive is safe — it will exit early.

---
## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone --quiet --depth 1 https://github.com/tsereda/CTDenoiser.git /content/CTDenoiser 2>/dev/null \
    || git -C /content/CTDenoiser pull --quiet
!pip install -q -e /content/CTDenoiser
!pip install -q tcia_utils pydicom h5py

---
## Configuration

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
DRIVE_CACHE   = '/content/drive/MyDrive/ldct_cache.h5'   # final destination
LOCAL_CACHE   = '/content/ldct_cache.h5'                 # fast scratch disk
DICOM_DIR     = '/content/ldct_dicom'                    # scratch for DICOMs

MAX_PATIENTS  = 10          # each ~4 GB; set None to grab all ~50
LOW_DOSE_TAG  = 'Low Dose Images'
FULL_DOSE_TAG = 'Full Dose Images'
BODY_PART     = 'ABDOMEN'
# ────────────────────────────────────────────────────────────────────────────

---
## Skip check

If the cache already exists on Drive the notebook stops here — nothing is re-downloaded or re-built.

In [ ]:
from pathlib import Path

if Path(DRIVE_CACHE).exists():
    import h5py
    with h5py.File(DRIVE_CACHE, 'r') as hf:
        keys = sorted(hf.keys())
        patients = sorted({k.rsplit('_', 1)[0] for k in keys})
    print(f'Cache already exists at {DRIVE_CACHE}')
    print(f'  {len(patients)} patients: {patients}')
    print(f'  {len(keys)} datasets: {keys[:6]}...')
    print()
    print('Nothing to do. Open ctdenoiser_colab.ipynb to start training.')
    print('Delete the file first if you want to rebuild from scratch.')
    raise SystemExit(0)

---
## Download TCIA LDCT data

In [ ]:
from tcia_utils import nbia
import pandas as pd

print('Querying TCIA for LDCT-and-Projection-data series...')
series = nbia.getSeries(collection='LDCT-and-Projection-data')
df = pd.DataFrame(series)

print(f'Total series: {len(df)}')
print('Body parts:', df['BodyPartExamined'].unique())
print('Descriptions:', df['SeriesDescription'].unique()[:10])

In [ ]:
imgs = df[
    (df['BodyPartExamined'] == BODY_PART) &
    (df['SeriesDescription'].isin([LOW_DOSE_TAG, FULL_DOSE_TAG]))
].copy()

by_patient = imgs.groupby('PatientID')['SeriesDescription'].apply(set)
paired_ids = by_patient[
    by_patient.apply(lambda x: LOW_DOSE_TAG in x and FULL_DOSE_TAG in x)
].index.tolist()

print(f'Patients with paired low+full dose: {len(paired_ids)}')
print('Sample IDs:', paired_ids[:5])

if MAX_PATIENTS:
    paired_ids = paired_ids[:MAX_PATIENTS]
    print(f'Limiting to {MAX_PATIENTS} patients.')

paired_series = imgs[imgs['PatientID'].isin(paired_ids)]
print(f'Series to download: {len(paired_series)}')
paired_series[['PatientID', 'SeriesDescription', 'SeriesInstanceUID']].head(10)

In [ ]:
import os

os.makedirs(DICOM_DIR, exist_ok=True)
print(f'Downloading {len(paired_series)} series to {DICOM_DIR}...')

nbia.downloadSeries(
    series_data=paired_series['SeriesInstanceUID'].tolist(),
    path=DICOM_DIR,
    input_type='list'
)
print('Download complete.')

In [ ]:
import glob

dcm_files = glob.glob(os.path.join(DICOM_DIR, '**', '*.dcm'), recursive=True)
print(f'Total DICOM files found: {len(dcm_files)}')

series_dirs = sorted(set(os.path.dirname(f) for f in dcm_files))
print(f'Series folders: {len(series_dirs)}')
for d in series_dirs[:6]:
    n = len(glob.glob(os.path.join(d, '*.dcm')))
    print(f'  {os.path.basename(d)[:60]}  ({n} slices)')

---
## Build HDF5 cache

In [ ]:
import numpy as np
import h5py
from pathlib import Path
from ctdenoiser.data.dicom import read_series_hu

uid_meta = {
    row['SeriesInstanceUID']: (row['PatientID'], row['SeriesDescription'])
    for _, row in paired_series.iterrows()
}

series_dirs = [d for d in Path(DICOM_DIR).iterdir() if d.is_dir()]

print(f'Building HDF5 cache at {LOCAL_CACHE} ...')
failed = []
with h5py.File(LOCAL_CACHE, 'w') as hf:
    for sdir in sorted(series_dirs):
        uid = sdir.name
        if uid not in uid_meta:
            continue
        pid, desc = uid_meta[uid]
        dose_key = 'low' if desc == LOW_DOSE_TAG else 'full'
        h5_key   = f'{pid}_{dose_key}'
        try:
            arr = read_series_hu(str(sdir))
            hf.create_dataset(h5_key, data=arr, compression='lzf')
            print(f'  wrote {h5_key:30s}  {arr.shape}  [{arr.min():.0f}, {arr.max():.0f}] HU')
        except Exception as e:
            print(f'  SKIP {h5_key}: {e}')
            failed.append(h5_key)

print(f'\nDone. Failed: {failed if failed else "none"}')

In [ ]:
with h5py.File(LOCAL_CACHE, 'r') as hf:
    keys = sorted(hf.keys())
    patients = sorted({k.rsplit('_', 1)[0] for k in keys})
    print(f'{len(patients)} patients, {len(keys)} datasets')
    print('Patients:', patients)
    print()
    for k in keys[:6]:
        print(f'  {k:30s}  {hf[k].shape}')

In [ ]:
import shutil

print(f'Copying {LOCAL_CACHE} -> {DRIVE_CACHE} ...')
shutil.copy(LOCAL_CACHE, DRIVE_CACHE)
print('Saved to Drive.')
print()
print('Cache is ready. Open ctdenoiser_colab.ipynb to start training.')